# Lab 08: RNN Text Classification — IMDB Movie Reviews

**Course:** Artificial Neural Networks and Deep Learning  
**Model:** SimpleRNN for binary sentiment classification  
**Dataset:** IMDB Movie Reviews

### Aim
Implement and evaluate an RNN-based binary text classifier using TensorFlow/Keras. The assignment requires an Embedding layer, SimpleRNN layer, and sigmoid output layer, with vocabulary size 10,000 and sequence length 200.


## 1. Import Libraries and Set Seed

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


## 2. Task 1 — Load and Explore the Dataset

The assignment specifies a vocabulary size of **10,000** and maximum sequence length of **200**. The IMDB dataset contains binary labels: `0 = Negative` and `1 = Positive`.


In [ ]:
VOCAB_SIZE = 10_000
MAX_LEN = 200

(x_train_raw, y_train), (x_test_raw, y_test) = tf.keras.datasets.imdb.load_data(
    num_words=VOCAB_SIZE
)

print("Number of training samples:", len(x_train_raw))
print("Number of testing samples:", len(x_test_raw))
print("Training data shape before padding:", np.array(x_train_raw, dtype=object).shape)
print("Testing data shape before padding:", np.array(x_test_raw, dtype=object).shape)

print("\nTraining positive reviews:", np.sum(y_train == 1))
print("Training negative reviews:", np.sum(y_train == 0))
print("Testing positive reviews:", np.sum(y_test == 1))
print("Testing negative reviews:", np.sum(y_test == 0))

print("\nFirst review (encoded):")
print(x_train_raw[0][:30])
print("First review length:", len(x_train_raw[0]))


## 3. Task 2 — Preprocess and Pad the Reviews

In [ ]:
x_train = tf.keras.utils.pad_sequences(
    x_train_raw, maxlen=MAX_LEN, padding="post", truncating="post"
)
x_test = tf.keras.utils.pad_sequences(
    x_test_raw, maxlen=MAX_LEN, padding="post", truncating="post"
)

print("Final training shape:", x_train.shape)
print("Final testing shape:", x_test.shape)


## 4. Task 3 — Build the RNN Classifier

Architecture:

**Input → Embedding(128) → SimpleRNN(64) → Dense(1, sigmoid)**


In [ ]:
def build_rnn(rnn_units=64):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(MAX_LEN,)),
        tf.keras.layers.Embedding(VOCAB_SIZE, 128),
        tf.keras.layers.SimpleRNN(rnn_units),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])
    return model

model = build_rnn(64)
model.summary()


## 5. Task 4 — Compile and Train

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.20,
    verbose=1
)


In [ ]:
print("Final Training Accuracy:", history.history["accuracy"][-1])
print("Final Validation Accuracy:", history.history["val_accuracy"][-1])
print("Final Training Loss:", history.history["loss"][-1])
print("Final Validation Loss:", history.history["val_loss"][-1])


## 6. Task 5 — Test Evaluation

The exact accuracy is generated when the notebook is executed. It should not be hard-coded because it depends on the actual training run and environment.


In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")


### Q1. What is the final test accuracy?

Run the previous cell and report the printed **Test Accuracy** value in your lab report.

### Q2. Is test accuracy higher or lower than training accuracy?

In a typical run, test accuracy is lower than training accuracy because the test set contains unseen reviews. The model optimizes its parameters using the training data, so it generally performs best on that data. A large gap between training and test/validation accuracy can indicate overfitting.


## 7. Task 6 — Accuracy Graph

In [ ]:
epochs = range(1, len(history.history["accuracy"]) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, history.history["accuracy"], marker="o", label="Training Accuracy")
plt.plot(epochs, history.history["val_accuracy"], marker="o", label="Validation Accuracy")
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()


## 8. Task 6 — Loss Graph

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(epochs, history.history["loss"], marker="o", label="Training Loss")
plt.plot(epochs, history.history["val_loss"], marker="o", label="Validation Loss")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()


### Graph Interpretation

Training accuracy should generally increase and training loss should generally decrease as the model learns. If validation accuracy stops improving while training accuracy continues increasing, or validation loss begins increasing while training loss keeps decreasing, the model may be overfitting.

The validation curves are therefore important for deciding whether additional training is beneficial.


## 9. Task 7 — Experiment A: RNN Units

Compare **32, 64, and 128** RNN units. Each experiment uses 5 epochs and batch size 64.


In [ ]:
def run_experiment(rnn_units=64, epochs=5, batch_size=64):
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)

    m = build_rnn(rnn_units)
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

    h = m.fit(
        x_train, y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=0.20,
        verbose=0
    )

    loss, acc = m.evaluate(x_test, y_test, verbose=0)

    return {
        "Training Accuracy": h.history["accuracy"][-1],
        "Validation Accuracy": h.history["val_accuracy"][-1],
        "Test Accuracy": acc
    }

rows = []
for units in [32, 64, 128]:
    r = run_experiment(rnn_units=units, epochs=5, batch_size=64)
    rows.append({"Experiment": f"A — RNN = {units}", **r})

experiment_A = pd.DataFrame(rows)
experiment_A


## 10. Experiment B — Number of Epochs

Compare **3, 5, and 10 epochs**, keeping RNN units at 64 and batch size at 64.


In [ ]:
rows = []
for ep in [3, 5, 10]:
    r = run_experiment(rnn_units=64, epochs=ep, batch_size=64)
    rows.append({"Experiment": f"B — Epochs = {ep}", **r})

experiment_B = pd.DataFrame(rows)
experiment_B


## 11. Experiment C — Batch Size

Compare **32, 64, and 128**, keeping RNN units at 64 and epochs at 5.


In [ ]:
rows = []
for batch in [32, 64, 128]:
    r = run_experiment(rnn_units=64, epochs=5, batch_size=batch)
    rows.append({"Experiment": f"C — Batch = {batch}", **r})

experiment_C = pd.DataFrame(rows)
experiment_C


## 12. Complete Results Table

In [ ]:
results_table = pd.concat(
    [experiment_A, experiment_B, experiment_C],
    ignore_index=True
)

results_table


### Comparison

The best configuration should be selected primarily from **validation and test accuracy**, not training accuracy alone.

- More RNN units increase model capacity but also increase computation and parameter count.
- More epochs can improve performance initially, but excessive training may cause overfitting.
- Batch size affects the optimization process and training efficiency.
- The experimental tables above provide the actual values from your run.


## 13. Task 8 — Text Prediction

The following function accepts a new movie review and predicts **Positive** or **Negative**. Five original reviews are tested as required by the assignment.


In [ ]:
word_index = tf.keras.datasets.imdb.get_word_index()

def encode_review(text):
    tokens = text.lower().split()
    encoded = []

    for word in tokens:
        # Remove simple punctuation around words.
        word = word.strip(".,!?;:'\"()[]{}")
        idx = word_index.get(word, 2) + 3
        if idx >= VOCAB_SIZE:
            idx = 2
        encoded.append(idx)

    return tf.keras.utils.pad_sequences(
        [encoded], maxlen=MAX_LEN, padding="post", truncating="post"
    )

def predict_sentiment(review):
    probability = float(model.predict(encode_review(review), verbose=0)[0][0])
    sentiment = "Positive" if probability >= 0.5 else "Negative"
    return sentiment, probability


In [ ]:
reviews = [
    ("I loved this movie. The story was exciting, emotional and beautifully acted.", "Positive"),
    ("This movie was painfully boring and the acting was terrible.", "Negative"),
    ("An excellent film with memorable characters and a wonderful ending.", "Positive"),
    ("I hated this movie because the plot was confusing and disappointing.", "Negative"),
    ("Fantastic performances and a great story. I would definitely watch it again.", "Positive")
]

prediction_results = []

for i, (review, expected) in enumerate(reviews, 1):
    predicted, probability = predict_sentiment(review)
    prediction_results.append({
        "Review": f"Review {i}",
        "Actual/Expected Sentiment": expected,
        "Predicted Sentiment": predicted,
        "Probability": round(probability, 4)
    })

pd.DataFrame(prediction_results)


## 14. Task 9 — Challenge: Two-Layer RNN

Architecture:

**Embedding → SimpleRNN(return_sequences=True) → SimpleRNN → Dense**

`return_sequences=True` is required in the first RNN layer because the second RNN needs a sequence of outputs from the first RNN.


In [ ]:
def build_two_layer_rnn():
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(MAX_LEN,)),
        tf.keras.layers.Embedding(VOCAB_SIZE, 128),
        tf.keras.layers.SimpleRNN(64, return_sequences=True),
        tf.keras.layers.SimpleRNN(64),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

two_layer_model = build_two_layer_rnn()
two_layer_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

two_layer_history = two_layer_model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.20,
    verbose=1
)

two_loss, two_accuracy = two_layer_model.evaluate(x_test, y_test, verbose=0)

print(f"Single-layer RNN test accuracy: {test_accuracy:.4f}")
print(f"Two-layer RNN test accuracy:    {two_accuracy:.4f}")


### Challenge Answers

**1. Which model gives better test accuracy?**  
Compare the two printed values. The model with the higher test accuracy performs better for this run.

**2. Which model takes longer to train?**  
The two-layer RNN normally takes longer because it performs recurrent computation through an additional RNN layer.

**3. Does adding another RNN layer always improve performance?**  
No. A deeper model can learn more complex representations, but it can also increase computational cost and overfitting. More layers do not guarantee better test accuracy.

**4. What is the purpose of `return_sequences=True`?**  
It makes the first RNN return the hidden-state output at every timestep. This sequence becomes the input to the second RNN layer.


## 15. Discussion Questions

### 1. What is an RNN and why is it suitable for text classification?
An RNN is a neural network designed for sequential data. It maintains a hidden state that carries information from previous timesteps, making it suitable for text where word order and context matter.

### 2. What is the purpose of the Embedding layer?
It converts integer word IDs into dense trainable vectors. During training, these vectors learn useful representations of words.

### 3. Why is padding required?
Reviews have different lengths. Padding makes all sequences the same length so they can be processed together in batches.

### 4. Why is sigmoid activation used in the output layer?
The task has two classes. Sigmoid produces a value between 0 and 1 that can represent the probability of the positive class.

### 5. Why is binary cross-entropy used?
Binary cross-entropy is designed for binary classification and measures the difference between the predicted probability and the true binary label.

### 6. What is the vanishing-gradient problem?
During backpropagation through long sequences, gradients can become extremely small. This makes it difficult for a basic RNN to learn relationships between distant timesteps.

### 7. What happens when the number of RNN units is increased?
The model gains more representational capacity, but it also has more parameters and requires more computation. Increasing units does not always improve generalization.

### 8. What is overfitting?
Overfitting occurs when a model performs very well on training data but poorly on unseen data because it has learned patterns that do not generalize.

### 9. How can overfitting be reduced in an RNN?
Use dropout, early stopping, appropriate model size, more training data, and careful tuning of epochs and other hyperparameters.

### 10. What is the difference between SimpleRNN, LSTM and GRU?
SimpleRNN is the basic recurrent architecture. LSTM uses gates and a cell state to better preserve long-term information. GRU also uses gates but has a simpler structure and generally fewer parameters than LSTM.


## 16. Final Conclusion

This laboratory demonstrated the complete workflow for binary sentiment classification using a Recurrent Neural Network on the IMDB movie-review dataset. The reviews were loaded as integer sequences, restricted to the 10,000 most frequent words, and padded or truncated to a fixed length of 200. An Embedding layer transformed the word indices into trainable vector representations, while a SimpleRNN learned sequential patterns before a sigmoid output classified each review as positive or negative.

The model was trained using Adam, binary cross-entropy, a batch size of 64, and a 20% validation split. Accuracy and loss graphs were used to examine learning and possible overfitting. Experiments with different RNN sizes, numbers of epochs, and batch sizes demonstrated the effect of hyperparameter choices on performance. A two-layer RNN was also implemented to compare a deeper recurrent architecture with the basic model.

Overall, the experiment shows that SimpleRNN can perform useful sentiment classification, while LSTM or GRU architectures may be preferable when learning long-range dependencies in longer text sequences.
